# Bagging : solution de l'exercice

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/ml-classique-et-deep-learning/bagging/bagging-solution.ipynb)

📖 **Le mini-cours complet** : [Comprendre enfin pourquoi le bagging marche (et pas seulement comment)](https://www.monshotdata.com/p/bagging)

🧭 **Le cours** : [ML classique et Deep Learning › 2.1 Fondations mathématiques](https://www.monshotdata.com/mini-cours/ml-classique-et-deep-learning#h2sc21)

---

Voici une solution de référence de l'exercice sur le sous-échantillonnage de features. **Essaie d'abord de le faire toi-même** dans `bagging-exercice.ipynb`, tu apprendras beaucoup plus. Ici, la solution est testée sur un dataset de **10 features**, là où le sous-échantillonnage de features a du sens.

**À la fin de ce notebook, tu sauras :**

- coder le sous-échantillonnage de colonnes et de lignes avec pandas ;
- entraîner une forêt « maison » et la comparer à un arbre seul et à `RandomForestClassifier` ;
- observer l'effet du sous-échantillonnage de features en haute dimension (il reste souvent modeste : l'intérêt est la décorrélation des arbres).


### Avant de lancer

Aucun fichier externe, aucun GPU : **Exécution → Tout exécuter** (environ une minute, le bootstrap de lignes est volontairement écrit « à la main »).

🧪 *Testé le 20 novembre 2026 avec Python 3.13, NumPy, pandas et scikit-learn dans leurs versions de Colab.*

## 1. La solution

Une seule fonction à retenir. Points d'attention :
- la colonne cible `y` est **exclue** du tirage de colonnes, puis rajoutée à la fin ;
- les colonnes dupliquées reçoivent un suffixe `__dup` pour éviter les noms en double ;
- chaque arbre mémorise (`cols_`) les colonnes d'origine qu'il a vues, pour pouvoir prédire sur de nouvelles données.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def sampling_function(df, row_sample_ratio, feature_sample_ratio):
    """Retourne un dataset de même taille que df, avec des lignes ET des colonnes dupliquées."""
    features = [c for c in df.columns if c != "y"]
    total_samples = df.shape[0]
    final_size = total_samples

    # --- Colonnes : échantillon initial (sans y), puis duplications ---
    initial_column_size = max(1, int(feature_sample_ratio * len(features)))
    initial_column_sample_df = df[features].sample(n=initial_column_size, axis=1)

    new_columns = pd.DataFrame(index=df.index)
    for j in range(len(features) - initial_column_size):
        sampled_column = initial_column_sample_df.sample(n=1, axis=1)
        base_name = sampled_column.columns[0]
        new_columns[f"{base_name}__dup{j}"] = sampled_column.iloc[:, 0]

    column_sampled_df = pd.concat((initial_column_sample_df, new_columns, df[["y"]]), axis=1)

    # --- Lignes : échantillon initial, puis duplications ---
    initial_sample_size = int(row_sample_ratio * total_samples)
    initial_row_sample_df = column_sampled_df.sample(n=initial_sample_size)
    new_rows = column_sampled_df.sample(n=final_size - initial_sample_size, replace=True)
    return pd.concat((initial_row_sample_df, new_rows))

def predict(models, test_data):
    all_preds = [m.predict(test_data[m.cols_].values) for m in models]
    return np.array([Counter(p[i] for p in all_preds).most_common(1)[0][0] for i in range(len(test_data))])

## 2. Entraîner la forêt sur un dataset en 10 dimensions

In [2]:
X, y = make_classification(n_samples=1200, n_features=10, n_informative=4, n_redundant=2,
                           n_clusters_per_class=1, class_sep=0.5, random_state=21)
cols = [f"X{i+1}" for i in range(X.shape[1])]
df_train = pd.DataFrame(X[:1000], columns=cols); df_train["y"] = y[:1000]
df_test  = pd.DataFrame(X[1000:], columns=cols); df_test["y"]  = y[1000:]

np.random.seed(42)   # le tirage des lignes et des colonnes utilise le hasard de NumPy : on le fixe pour retrouver les mêmes résultats
all_trees = []
for _ in range(21):
    data = sampling_function(df_train, row_sample_ratio=0.5, feature_sample_ratio=0.5)
    features = [c for c in data.columns if c != "y"]
    model = DecisionTreeClassifier(max_features="sqrt").fit(data[features].values, data["y"])
    model.cols_ = [c.split("__dup")[0] for c in features]   # colonnes d'origine vues par cet arbre
    all_trees.append(model)

## 3. Comparer aux modèles de référence

Avec la graine fixée plus haut, les trois précisions sont proches (0,92 à 0,93). Sur seulement 200 exemples de test, un écart de quelques millièmes n'est pas significatif : ne le sur-interprète pas. Change la graine et tu verras le classement bouger. Le vrai atout d'une forêt n'est pas la précision d'un tirage, mais sa **stabilité** d'un tirage à l'autre.

In [3]:
dtree = DecisionTreeClassifier(random_state=0).fit(df_train[cols], df_train["y"])
rf = RandomForestClassifier(max_features="sqrt", max_samples=0.5, n_estimators=21, random_state=0).fit(df_train[cols], df_train["y"])
custom_acc = (predict(all_trees, df_test[cols]) == df_test["y"].values).mean()

print(f"Arbre de décision seul         : {dtree.score(df_test[cols], df_test['y']):.3f}")
print(f"Random Forest (scikit-learn)   : {rf.score(df_test[cols], df_test['y']):.3f}")
print(f"Random Forest personnalisée    : {custom_acc:.3f}")

Arbre de décision seul         : 0.920
Random Forest (scikit-learn)   : 0.930
Random Forest personnalisée    : 0.925


---

## Pour aller plus loin

- [Comprendre enfin pourquoi le bagging marche (et pas seulement comment)](https://www.monshotdata.com/p/bagging) — le mini-cours complet et le notebook de démonstration